In [20]:
# ==============================================
# 📘 Airbnb Data Warehouse Project - ETL Notebook
# ==============================================
# Author: Shazil
# Description: End-to-end ETL pipeline for Airbnb Listings dataset
# Steps:
# 1. Extract - Load dataset
# 2. Transform - Clean, validate, and map to Star Schema
# 3. Load - Store in MySQL Database
# ==============================================


In [37]:
!pip install cryptography

  Using cached cryptography-46.0.3-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
Using cached cryptography-46.0.3-cp311-abi3-win_amd64.whl (3.5 MB)
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Uninstalling cffi-1.17.1:
      Successfully uninstalled cffi-1.17.1



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install pymysql

  Using cached pymysql-1.1.2-py3-none-any.whl.metadata (4.3 kB)
Using cached pymysql-1.1.2-py3-none-any.whl (45 kB)



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [22]:
# Load dataset from local CSV file
# (Update the path to your Kaggle CSV)
file_path = "../data/AB_NYC.csv"
df = pd.read_csv(file_path)

# View top records
print("✅ Dataset Loaded Successfully!")
print("Total Records:", len(df))
df.head()

✅ Dataset Loaded Successfully!
Total Records: 48895


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [23]:
# ==============================================
# 🟩 2. TRANSFORM PHASE
# ==============================================

# --- 2.1 Handle Missing Values ---
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')
df['reviews_per_month'].fillna(0, inplace=True)
df['host_name'].fillna("Unknown", inplace=True)
df['name'].fillna("No Title", inplace=True)
df['neighbourhood_group'].fillna("Unknown", inplace=True)
df['neighbourhood'].fillna("Unknown", inplace=True)

# --- 2.2 Remove Duplicates ---
df.drop_duplicates(subset=['id'], inplace=True)

# --- 2.3 Validate Numeric Fields ---
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['minimum_nights'] = pd.to_numeric(df['minimum_nights'], errors='coerce')
df['availability_365'] = pd.to_numeric(df['availability_365'], errors='coerce')

# Replace invalid or negative values
df['price'] = df['price'].clip(lower=0)
df['minimum_nights'] = df['minimum_nights'].clip(lower=0)
df['availability_365'] = df['availability_365'].clip(lower=0)

C:\Users\Muhammad Shazil\AppData\Local\Temp\ipykernel_21240\1041333568.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['reviews_per_month'].fillna(0, inplace=True)
C:\Users\Muhammad Shazil\AppData\Local\Temp\ipykernel_21240\1041333568.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves 

In [24]:
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaT,0.00,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [ ]:
# --- 2.4 Create Dimension Tables ---

# Dim Listing
dim_listing = df[['id', 'name']].drop_duplicates().reset_index(drop=True)
dim_listing.rename(columns={'id': 'listing_id', 'name': 'listing_name'}, inplace=True)

# Dim Host
dim_host = df[['host_id', 'host_name']].drop_duplicates().reset_index(drop=True)

# Dim Location
dim_location = df[['neighbourhood_group', 'neighbourhood', 'latitude', 'longitude']].drop_duplicates().reset_index(drop=True)
dim_location['neighbourhood_id'] = dim_location.index + 1

# Dim Room Type
dim_room = df[['room_type']].drop_duplicates().reset_index(drop=True)
dim_room['room_type_id'] = dim_room.index + 1

# Dim Date
dim_date = df[['last_review']].drop_duplicates().reset_index(drop=True)
dim_date['date_id'] = dim_date.index + 1
dim_date['year'] = dim_date['last_review'].dt.year
dim_date['month'] = dim_date['last_review'].dt.month
dim_date['day'] = dim_date['last_review'].dt.day
dim_date['weekday'] = dim_date['last_review'].dt.day_name()

# --- 2.5 Create Fact Table ---
fact_listing = (
    df.merge(dim_listing, left_on='id', right_on='listing_id')
      .merge(dim_host, on=['host_id', 'host_name'], how='left')
      .merge(dim_location, on=['neighbourhood_group', 'neighbourhood', 'latitude', 'longitude'], how='left')
      .merge(dim_room, on=['room_type'], how='left')
      .merge(dim_date, on=['last_review'], how='left')
      [['listing_id', 'host_id', 'neighbourhood_id', 'room_type_id', 'date_id',
        'price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month',
        'calculated_host_listings_count', 'availability_365']]
)

In [ ]:
# Show samples
print("\n✅ Transformation Completed!")
print("Fact Table shape:", fact_listing.shape)
fact_listing.head()


✅ Transformation Completed!
Fact Table shape: (48895, 11)


,listing_id,host_id,neighbourhood_id,room_type_id,date_id,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,2787,1,1,1,149,1,9,0.21,6,365
1,2595,2845,2,2,2,225,1,45,0.38,2,355
2,3647,4632,3,1,3,150,3,0,0.00,1,365
3,3831,4869,4,2,4,89,1,270,4.64,1,194
4,5022,7192,5,2,5,80,10,9,0.10,1,0


In [ ]:
dim_date.head()
# dim_date.shape

,last_review,date_id,year,month,day,weekday
0,2018-10-19,1,2018.0,10.0,19.0,Friday
1,2019-05-21,2,2019.0,5.0,21.0,Tuesday
2,NaT,3,NaN,NaN,NaN,NaN
3,2019-07-05,4,2019.0,7.0,5.0,Friday
4,2018-11-19,5,2018.0,11.0,19.0,Monday
...,...,...,...,...,...,...
1755,2018-01-28,1756,2018.0,1.0,28.0,Sunday
1756,2017-10-26,1757,2017.0,10.0,26.0,Thursday
1757,2018-01-22,1758,2018.0,1.0,22.0,Monday
1758,2017-11-16,1759,2017.0,11.0,16.0,Thursday


In [39]:
dim_location.head()

,neighbourhood_group,neighbourhood,latitude,longitude,neighbourhood_id
0,Brooklyn,Kensington,40.64749,-73.97237,1
1,Manhattan,Midtown,40.75362,-73.98377,2
2,Manhattan,Harlem,40.80902,-73.94190,3
3,Brooklyn,Clinton Hill,40.68514,-73.95976,4
4,Manhattan,East Harlem,40.79851,-73.94399,5


In [38]:
# --- 3.1 Connect to MySQL ---
# Update credentials for your local setup
user = 'root'
password = 'shazilkhan0318$'
host = 'localhost'
port = '3306'
database = 'airbnb_dwh'

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")

# --- 3.2 Load DataFrames to Database ---
tables = {
    'dim_listing': dim_listing,
    'dim_host': dim_host,
    'dim_location': dim_location,
    'dim_room': dim_room,
    'dim_date': dim_date,
    'fact_listing': fact_listing
}

for name, table in tables.items():
    table.to_sql(name, engine, index=False, if_exists='replace')
    print(f"✅ Loaded {name} ({len(table)} rows)")

print("\n🎉 Data successfully loaded into MySQL in Star Schema format!")


✅ Loaded dim_listing (48895 rows)
✅ Loaded dim_host (37457 rows)
✅ Loaded dim_location (48871 rows)
✅ Loaded dim_room (3 rows)
✅ Loaded dim_date (1765 rows)
✅ Loaded fact_listing (48895 rows)

🎉 Data successfully loaded into MySQL in Star Schema format!
